In [ ]:
#Essa seção do Notebook é focada na preparação do schema Gold
#Execute a Silver antes da Gold para consumir os dados já tratados.
#A carga sobrescreve as tabelas; execute as células na ordem para reconstruir as relações.

from pyspark.sql import functions as F #Importa as funções utilizadas nas transformações dos DataFrames.
from pyspark.sql.window import Window #Importa janelas para ordenar registros, gerar chaves e aplicar critérios de seleção.

spark.sql("CREATE SCHEMA IF NOT EXISTS medallion.gold") #Cria o schema de destino caso ele ainda não exista.
print("Schema Gold preparado") #Informa a conclusão desta etapa ou o valor utilizado no processamento.


In [ ]:
#Essa seção do Notebook é focada em dimensão de filmes

#A dimensão mantém os metadados de todos os filmes; a seleção de lançados ocorre na fato.
#A chave substituta permite relacionar as tabelas sem usar o ID da origem em cada relação.
#As chaves geradas por row_number podem mudar entre cargas; por isso as tabelas dependentes são reconstruídas.
#A ordenação global pode concentrar o processamento em uma partição.

janela = Window.orderBy("id_filme") #Define a ordem usada pela função de janela.

df_dim_filmes = ( #Cria o DataFrame da dimensão de filmes com os atributos definidos para a Gold.
    spark.table("medallion.silver.tb_info_filmes") #Lê a tabela persistida que serve de origem para este bloco.
    .withColumn("id_filme", F.col("id_filme").cast("string")) #Garante a representação textual prevista no esquema.
    .withColumn("data_lancamento", F.col("data_lancamento").cast("date")) #Mantém somente a data no tipo DATE.
    .withColumn("ano_lancamento", F.col("ano_lancamento").cast("int")) #Aplica o tipo inteiro exigido para o atributo ou contagem.
    .withColumn("duracao_minutos", F.col("duracao_minutos").cast("int")) #Aplica o tipo inteiro exigido para o atributo ou contagem.
    .withColumn(
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        F.row_number().over(janela).cast("bigint") #Gera a chave substituta sequencial e converte para BIGINT, conforme o esquema exigido.
    )
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "titulo", #Título do filme utilizado na identificação dos resultados de negócio.
        "data_lancamento", #Data de lançamento utilizada nos filtros temporais das análises.
        "ano_lancamento", #Ano de lançamento utilizado na descrição e no agrupamento dos filmes.
        "duracao_minutos", #Duração do filme expressa em minutos.
        "idioma_original", #Idioma original do filme, preservado como atributo descritivo.
        "status_filme", #Situação do filme, utilizada para selecionar somente os lançados na fato.
        "sinopse" #Descrição do filme utilizada na composição do documento para IA.
    )
)

assert ( #Interrompe a execução se a condição de qualidade abaixo não for atendida.
    df_dim_filmes.count()
    == df_dim_filmes.select("sk_movie_id").distinct().count() #Compara a quantidade total com as chaves distintas para identificar duplicações.
), "dim_movies possui chaves substitutas duplicadas"

(
    df_dim_filmes.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.dim_movies") #Publica os dados na tabela medallion.gold.dim_movies.
)

print("✓ gold.dim_movies concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em dimensão de gêneros

#Seleciona apenas o gênero para que cada nome receba uma única chave, independentemente do número de filmes.

df_generos = spark.table("medallion.silver.tb_generos") #Lê a tabela persistida que serve de origem para este bloco.

janela_generos = Window.orderBy("nome_genero") #Define a ordem usada pela função de janela.

df_dim_generos = ( #Cria o catálogo de gêneros com uma chave substituta por nome.
    df_generos
    .select("nome_genero") #Seleciona os atributos necessários para esta relação ou resultado.
    .distinct() #Elimina gêneros repetidos entre diferentes filmes antes de gerar as chaves.
    .withColumn(
        "sk_genre_id", #Chave substituta que identifica o gênero na dimensão e na ponte.
        F.row_number().over(janela_generos).cast("bigint") #Gera a chave substituta sequencial e converte para BIGINT, conforme o esquema exigido.
    )
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_genre_id", #Chave substituta que identifica o gênero na dimensão e na ponte.
        "nome_genero" #Nome do gênero utilizado para agrupar e contar os filmes.
    )
)

(
    df_dim_generos.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.dim_genres") #Publica os dados na tabela medallion.gold.dim_genres.
)

print("✓ gold.dim_genres concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em relação entre filmes e gêneros

#Um filme pode ter vários gêneros; a ponte armazena essas relações sem aumentar as linhas da fato.
#Os joins internos mantêm apenas relações com filme e gênero cadastrados nas dimensões.

df_filmes_generos = ( #Cria as relações entre as chaves dos filmes e dos gêneros.
    spark.table("medallion.silver.tb_generos") #Lê a tabela persistida que serve de origem para este bloco.

    .join( #Busca a chave substituta do filme usando seu identificador original.
        df_dim_filmes.select("id_filme", "sk_movie_id"), #Seleciona os atributos necessários para esta relação ou resultado.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )

    .join( #Busca a chave substituta do gênero usando o nome cadastrado.
        df_dim_generos,
        "nome_genero", #Nome do gênero utilizado para agrupar e contar os filmes.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )

    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "sk_genre_id" #Chave substituta que identifica o gênero na dimensão e na ponte.
    )

    .dropDuplicates() #Mantém uma única linha para cada par de chaves da ponte.
)

(
    df_filmes_generos.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.bridge_movie_genre") #Publica os dados na tabela medallion.gold.bridge_movie_genre.
)

print("✓ gold.bridge_movie_genre concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em dimensão de pessoas

#A pessoa é identificada pelo nome e pelo tipo de atuação disponível na Silver.
#A mesma pessoa pode aparecer como ator e diretor; essas combinações recebem chaves diferentes.
#A fonte não possui identificador de pessoa para distinguir homônimos.

df_pessoas = ( #Seleciona as combinações distintas de nome e tipo das pessoas físicas.
    spark.table("medallion.silver.tb_pessoas_empresas") #Lê a tabela persistida que serve de origem para este bloco.
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")) #Mantém atores, diretores e roteiristas, separando as pessoas das produtoras.
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        F.col("nome_entidade").alias("nome_pessoa"), #Renomeia o atributo da Silver para o nome definido na dimensão.
        F.col("tipo_entidade").alias("tipo_pessoa") #Preserva o papel de atuação na dimensão de pessoas.
    )
    .distinct() #Elimina repetições de nome e tipo de atuação antes de criar a dimensão.
)

janela_pessoas = Window.orderBy("nome_pessoa", "tipo_pessoa") #Define a ordem usada pela função de janela.

df_dim_pessoas = ( #Cria a dimensão de pessoas e atribui uma chave para cada combinação.
    df_pessoas
    .withColumn(
        "sk_person_id", #Chave substituta que identifica a combinação de pessoa e tipo de atuação.
        F.row_number().over(janela_pessoas).cast("bigint") #Gera a chave substituta sequencial e converte para BIGINT, conforme o esquema exigido.
    )
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_person_id", #Chave substituta que identifica a combinação de pessoa e tipo de atuação.
        "nome_pessoa", #Nome da pessoa utilizado nas consultas de elenco e direção.
        "tipo_pessoa" #Papel de atuação que diferencia ator, diretor e roteirista.
    )
)

(
    df_dim_pessoas.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.dim_people") #Publica os dados na tabela medallion.gold.dim_people.
)

print("✓ gold.dim_people concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em relação entre filmes e pessoas

#A ponte representa cada participação por filme e por pessoa/tipo de atuação.
#Os joins internos exigem correspondência nas duas dimensões antes de gravar a relação.

df_filmes_pessoas = ( #Cria as relações entre filmes e pessoas sem incluir as produtoras.
    spark.table("medallion.silver.tb_pessoas_empresas") #Lê a tabela persistida que serve de origem para este bloco.
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")) #Mantém atores, diretores e roteiristas, separando as pessoas das produtoras.
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        F.col("nome_entidade").alias("nome_pessoa"), #Renomeia o atributo da Silver para o nome definido na dimensão.
        F.col("tipo_entidade").alias("tipo_pessoa") #Preserva o papel de atuação na dimensão de pessoas.
    )

    .join( #Obtém a chave do filme para registrar a participação na ponte.
        spark.table("medallion.gold.dim_movies") #Lê a tabela persistida que serve de origem para este bloco.
        .select("id_filme", "sk_movie_id"), #Seleciona os atributos necessários para esta relação ou resultado.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )

    .join( #Obtém a chave da pessoa considerando nome e papel de atuação juntos.
        spark.table("medallion.gold.dim_people"), #Lê a tabela persistida que serve de origem para este bloco.
        ["nome_pessoa", "tipo_pessoa"],
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )

    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "sk_person_id" #Chave substituta que identifica a combinação de pessoa e tipo de atuação.
    )
    .dropDuplicates() #Mantém uma única linha para cada par de chaves da ponte.
)

(
    df_filmes_pessoas.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.bridge_movie_person") #Publica os dados na tabela medallion.gold.bridge_movie_person.
)

print("✓ gold.bridge_movie_person concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em dimensão de produtoras

#Separa as produtoras das pessoas físicas para analisar o desempenho das empresas.
#Cada nome distinto de produtora recebe uma chave própria.

df_produtoras = ( #Seleciona os nomes distintos das entidades classificadas como produtoras.
    spark.table("medallion.silver.tb_pessoas_empresas") #Lê a tabela persistida que serve de origem para este bloco.
    .filter(F.col("tipo_entidade") == "Produtora") #Seleciona somente as entidades classificadas como produtoras.
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        F.col("nome_entidade").alias("nome_produtora") #Renomeia o atributo da Silver para o nome definido na dimensão.
    )
    .distinct() #Elimina nomes repetidos de produtoras antes de gerar as chaves.
)

janela_produtoras = Window.orderBy("nome_produtora") #Ordena pela medida de negócio para identificar o primeiro lugar e seus empates.

df_dim_produtoras = ( #Cria a dimensão de empresas com uma chave por nome de produtora.
    df_produtoras
    .withColumn(
        "sk_company_id", #Chave substituta da produtora, utilizada nas relações com os filmes.
        F.row_number().over(janela_produtoras).cast("bigint") #Gera a chave substituta sequencial e converte para BIGINT, conforme o esquema exigido.
    )
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_company_id", #Chave substituta da produtora, utilizada nas relações com os filmes.
        "nome_produtora" #Nome da empresa utilizado no agrupamento dos resultados financeiros.
    )
)

(
    df_dim_produtoras.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.dim_companies") #Publica os dados na tabela medallion.gold.dim_companies.
)

print("✓ gold.dim_companies concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em relação entre filmes e produtoras

#Um filme pode ter várias produtoras; a ponte mantém uma linha por relação, sem repetir a fato.
#A associação não informa o percentual financeiro de cada coprodutora.

df_filmes_produtoras = ( #Cria a ponte com as relações entre filmes e produtoras.
    spark.table("medallion.silver.tb_pessoas_empresas") #Lê a tabela persistida que serve de origem para este bloco.
    .filter(F.col("tipo_entidade") == "Produtora") #Seleciona somente as entidades classificadas como produtoras.
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        F.col("nome_entidade").alias("nome_produtora") #Renomeia o atributo da Silver para o nome definido na dimensão.
    )

    .join( #Obtém a chave do filme associado à produtora.
        spark.table("medallion.gold.dim_movies") #Lê a tabela persistida que serve de origem para este bloco.
        .select("id_filme", "sk_movie_id"), #Seleciona os atributos necessários para esta relação ou resultado.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )

    .join( #Obtém a chave da produtora a partir do nome cadastrado.
        spark.table("medallion.gold.dim_companies"), #Lê a tabela persistida que serve de origem para este bloco.
        "nome_produtora", #Nome da empresa utilizado no agrupamento dos resultados financeiros.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )

    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "sk_company_id" #Chave substituta da produtora, utilizada nas relações com os filmes.
    )
    .dropDuplicates() #Mantém uma única linha para cada par de chaves da ponte.
)

(
    df_filmes_produtoras.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.bridge_movie_company") #Publica os dados na tabela medallion.gold.bridge_movie_company.
)

print("✓ gold.bridge_movie_company concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em resumo das avaliações por filme

#O resumo possui uma linha por filme com avaliações; filmes sem avaliações não são criados neste bloco.
#A contagem inclui avaliações sem nota válida, enquanto a média ignora notas nulas.
#O nome dim_reviews segue a estrutura solicitada no projeto, embora armazene medidas agregadas.

df_reviews = ( #Agrupa as avaliações dos usuários em um resumo por filme.
    spark.table("medallion.silver.tb_avaliacoes_usuarios") #Lê a tabela persistida que serve de origem para este bloco.
    .groupBy("id_filme") #Agrupa os registros por filme para produzir uma linha de resumo por chave.
    .agg( #Calcula as medidas de resumo do conjunto ou dos grupos definidos.
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"), #Conta todas as avaliações do filme, mesmo quando a nota está nula.
        F.round(F.avg("nota_usuario"), 2) #Calcula a média das notas não nulas e arredonda para duas casas decimais.
         .cast("double") #Aplica o tipo DOUBLE exigido para esta medida.
         .alias("nota_media_usuarios")
    )

    .join( #Relaciona o resumo das avaliações ao filme cadastrado na dimensão.
        spark.table("medallion.gold.dim_movies") #Lê a tabela persistida que serve de origem para este bloco.
        .select("id_filme", "sk_movie_id"), #Seleciona os atributos necessários para esta relação ou resultado.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )
)

janela_reviews = Window.orderBy("sk_movie_id") #Define a ordem usada pela função de janela.

df_dim_avaliacoes = ( #Cria a tabela de resumos de avaliações com a chave do filme correspondente.
    df_reviews
    .withColumn(
        "sk_review_id", #Chave substituta do resumo de avaliações de cada filme.
        F.row_number().over(janela_reviews).cast("bigint") #Gera a chave substituta sequencial e converte para BIGINT, conforme o esquema exigido.
    )
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_review_id", #Chave substituta do resumo de avaliações de cada filme.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "qtd_avaliacoes_usuarios", #Quantidade de avaliações recebidas pelo filme, incluindo notas ausentes.
        "nota_media_usuarios" #Média das notas válidas dos usuários, arredondada para duas casas.
    )
)

(
    df_dim_avaliacoes.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.dim_reviews") #Publica os dados na tabela medallion.gold.dim_reviews.
)

print("✓ gold.dim_reviews concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em desempenho financeiro e engajamento

#O grão da fato é uma linha por filme com status Lançado.
#Financeiro e engajamento precisam estar deduplicados por id_filme na Silver para os joins não multiplicarem registros.
#Os joins à esquerda preservam os filmes lançados mesmo quando faltam métricas.
#Lucro e conversão para BRL são herdados da Silver, sem recalcular valores nesta etapa.

df_desempenho_filmes = ( #Monta a fato com uma linha por filme lançado e suas métricas.
    spark.table("medallion.gold.dim_movies") #Lê a tabela persistida que serve de origem para este bloco.
    .filter(F.col("status_filme") == "Lançado") #Restringe o conjunto aos filmes lançados.
    .select("sk_movie_id", "id_filme") #Seleciona os atributos necessários para esta relação ou resultado.

    .join( #Acrescenta orçamento, receita e lucro sem excluir filmes sem dados financeiros.
        spark.table("medallion.silver.tb_financeiro_filmes"), #Lê a tabela persistida que serve de origem para este bloco.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "left" #Mantém todas as linhas da esquerda, mesmo sem correspondência na direita.
    )

    .join( #Acrescenta as métricas TMDB e IMDb sem excluir filmes sem engajamento.
        spark.table("medallion.silver.tb_metricas_engajamento"), #Lê a tabela persistida que serve de origem para este bloco.
        "id_filme", #Identificador original do filme, mantido para rastrear o registro até a Silver.
        "left" #Mantém todas as linhas da esquerda, mesmo sem correspondência na direita.
    )

    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.

        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"), #Padroniza orcamento_usd como DECIMAL(18,2), sem recalcular o valor da Silver.
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"), #Padroniza receita_usd como DECIMAL(18,2), sem recalcular o valor da Silver.
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"), #Padroniza lucro_usd como DECIMAL(18,2), sem recalcular o valor da Silver.

        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"), #Padroniza orcamento_brl como DECIMAL(18,2), sem recalcular o valor da Silver.
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"), #Padroniza receita_brl como DECIMAL(18,2), sem recalcular o valor da Silver.
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"), #Padroniza lucro_brl como DECIMAL(18,2), sem recalcular o valor da Silver.

        F.col("popularidade").cast("double").alias("popularidade"), #Aplica o tipo DOUBLE exigido para esta medida.
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"), #Aplica o tipo DOUBLE exigido para esta medida.
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"), #Aplica o tipo inteiro exigido para o atributo ou contagem.
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"), #Aplica o tipo DOUBLE exigido para esta medida.
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb") #Aplica o tipo inteiro exigido para o atributo ou contagem.
    )
)

#Compara linhas e chaves distintas para detectar multiplicação do grão nos joins.
assert ( #Interrompe a execução se a condição de qualidade abaixo não for atendida.
    df_desempenho_filmes.count()
    == df_desempenho_filmes.select("sk_movie_id").distinct().count() #Compara a quantidade total com as chaves distintas para identificar duplicações.
), "fact_movies_performance possui filmes duplicados"

(
    df_desempenho_filmes.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.fact_movies_performance") #Publica os dados na tabela medallion.gold.fact_movies_performance.
)

print("✓ gold.fact_movies_performance concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada na criação do contexto dos filmes para IA
#Agrega os nomes antes do join para manter um documento por filme.
#Ordena o elenco alfabeticamente; a Silver não preserva a ordem de importância dos atores.
#Os textos de fallback impedem que campos nulos anulem a concatenação inteira.
#Mantém o template solicitado para todo o catálogo, inclusive filmes sem métricas na fato.
#O template usa a expressão lançado mesmo para outros status; esta anotação não altera a regra existente.

df_pessoas_filme = ( #Agrupa atores e diretores em listas textuais antes de montar o documento.
    spark.table("medallion.gold.bridge_movie_person") #Lê a tabela persistida que serve de origem para este bloco.
    .join( #Recupera nome e tipo das pessoas associadas a cada filme.
        spark.table("medallion.gold.dim_people"), #Lê a tabela persistida que serve de origem para este bloco.
        "sk_person_id", #Chave substituta que identifica a combinação de pessoa e tipo de atuação.
        "inner" #Mantém somente as linhas com correspondência nos dois lados.
    )
    .groupBy("sk_movie_id") #Agrupa os registros por filme para produzir uma linha de resumo por chave.
    .agg( #Calcula as medidas de resumo do conjunto ou dos grupos definidos.
        F.concat_ws( #Une os nomes agregados em uma única string com separador.
            ", ", #Separa os nomes por vírgula e espaço para tornar a lista legível.
            F.sort_array( #Ordena os nomes para produzir texto com sequência reproduzível.
                F.collect_set( #Reúne nomes sem repetições dentro do filme.
                    F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa")) #Define o texto alternativo para o campo ausente ou vazio.
                )
            )
        ).alias("atores"), #Nomeia a lista textual de atores agregada para cada filme.

        F.concat_ws( #Une os nomes agregados em uma única string com separador.
            ", ", #Separa os nomes por vírgula e espaço para tornar a lista legível.
            F.sort_array( #Ordena os nomes para produzir texto com sequência reproduzível.
                F.collect_set( #Reúne nomes sem repetições dentro do filme.
                    F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa")) #Define o texto alternativo para o campo ausente ou vazio.
                )
            )
        ).alias("diretores") #Nomeia a lista textual de diretores agregada para cada filme.
    )
)

df_contexto_ia = ( #Cria um documento textual por filme para posterior vetorização.
    spark.table("medallion.gold.dim_movies") #Lê a tabela persistida que serve de origem para este bloco.
    .join( #Acrescenta as métricas financeiras ao catálogo usado no documento de IA.
        spark.table("medallion.gold.fact_movies_performance"), #Lê a tabela persistida que serve de origem para este bloco.
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "left" #Mantém todas as linhas da esquerda, mesmo sem correspondência na direita.
    )
    .join( #Acrescenta o elenco e a direção já agrupados, evitando multiplicar os documentos.
        df_pessoas_filme,
        "sk_movie_id", #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
        "left" #Mantém todas as linhas da esquerda, mesmo sem correspondência na direita.
    )
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        F.col("id_filme").alias("movie_id"), #Mantém a chave natural para rastrear o documento até o filme de origem.
        F.col("titulo").alias("title"), #Expõe o título com o nome de coluna solicitado para a tabela de IA.

        F.concat( #Monta o trecho de texto que fará parte do documento de contexto.
            F.lit("O filme "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.coalesce(F.col("titulo"), F.lit("Título não informado")), #Usa um texto alternativo quando a informação estiver nula.
            F.lit(", lançado no ano de "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")), #Usa um texto alternativo quando a informação estiver nula.
            F.lit(", faturou "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.coalesce( #Usa um texto alternativo quando a informação estiver nula.
                F.concat(F.lit("US$ "), F.col("receita_usd").cast("string")), #Monta o trecho de texto que fará parte do documento de contexto.
                F.lit("valor não informado") #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            ),
            F.lit(" e teve um custo de "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.coalesce( #Usa um texto alternativo quando a informação estiver nula.
                F.concat(F.lit("US$ "), F.col("orcamento_usd").cast("string")), #Monta o trecho de texto que fará parte do documento de contexto.
                F.lit("valor não informado") #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            ),
            F.lit(". Estrelado por "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.when( #Define o texto alternativo para o campo ausente ou vazio.
                F.col("atores").isNull() | (F.col("atores") == ""), #Trata tanto ausência de registro quanto lista agregada vazia.
                F.lit("elenco não informado") #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            ).otherwise(F.col("atores")), #Utiliza o elenco agregado quando a lista está preenchida.
            F.lit(" e dirigido por "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.when( #Define o texto alternativo para o campo ausente ou vazio.
                F.col("diretores").isNull() | (F.col("diretores") == ""), #Trata tanto ausência de registro quanto lista agregada vazia.
                F.lit("diretor não informado") #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            ).otherwise(F.col("diretores")), #Utiliza os diretores agregados quando a lista está preenchida.
            F.lit(", o filme possui a seguinte sinopse: "), #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
            F.coalesce(F.col("sinopse"), F.lit("sinopse não informada")), #Usa um texto alternativo quando a informação estiver nula.
            F.lit(".") #Acrescenta o trecho fixo da frase ou o texto de fallback previsto para o campo.
        ).alias("llm_context_document") #Nomeia o texto completo que será consumido pelo processo de vetorização.
    )
)

(
    df_contexto_ia.write #Inicia a gravação do DataFrame calculado nesta seção.
    .format("delta") #Utiliza o formato Delta para persistir a tabela.
    .mode("overwrite") #Substitui os dados anteriores nesta carga completa.
    .option("overwriteSchema", "true") #Permite atualizar o schema da tabela durante a sobrescrita.
    .saveAsTable("medallion.gold.gold_genai_movies_context") #Publica os dados na tabela medallion.gold.gold_genai_movies_context.
)

print("✓ gold.gold_genai_movies_context concluída") #Informa que a gravação desta tabela foi concluída.


In [ ]:
#Essa seção do Notebook é focada em perguntas de negócio

#As consultas usam display para apresentar os resultados sem bibliotecas de gráficos.
#Os períodos de dois e cinco anos partem do lançamento realizado mais recente, conforme o enunciado.
#A receita total considera os filmes lançados presentes na fato.
#O lucro integral do filme é associado a cada produtora; os totais não representam rateio entre coprodutoras.

df_fato = spark.table("medallion.gold.fact_movies_performance") #Carrega a fato de desempenho usada nas somas e rankings.
df_filmes = spark.table("medallion.gold.dim_movies") #Carrega os títulos, datas e status usados para contextualizar as métricas.

# Data limite para os recortes de 2 e 5 anos:
# lançamento realizado mais recente, ignorando datas futuras.
data_limite = ( #Calcula o lançamento realizado mais recente da base para servir de limite das análises.
    df_filmes
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.current_date()) #Exclui datas futuras da definição do lançamento realizado mais recente.
    )
    .agg(F.max("data_lancamento")) #Obtém a data mais recente entre os lançamentos válidos para definir o período de análise.
    .first()[0] #Obtém o valor único da agregação para reutilizá-lo nos filtros de período.
)

if data_limite is None: #Evita executar recortes temporais sem uma data de referência válida.
    raise ValueError("Não há filmes lançados com data válida para os recortes temporais")
print(f"Data limite utilizada: {data_limite}") #Informa a conclusão desta etapa ou o valor utilizado no processamento.


# 1. Receita total em R$
display( #Exibe o resultado do DataFrame para responder à pergunta de negócio.
    df_fato.agg( #Calcula as medidas de resumo do conjunto ou dos grupos definidos.
        F.sum("receita_brl").alias("receita_total_brl") #Soma as receitas disponíveis em reais; valores nulos não contribuem para a soma.
    )
)


# 2. Top 5 filmes com maior popularidade
display( #Exibe o resultado do DataFrame para responder à pergunta de negócio.
    df_fato
    .join(df_filmes.select("sk_movie_id", "titulo"), "sk_movie_id") #Acrescenta o título aos filmes classificados por popularidade.
    .filter(F.col("popularidade").isNotNull()) #Retira filmes sem popularidade conhecida antes de selecionar os cinco maiores.
    .select("titulo", "popularidade") #Seleciona os atributos necessários para esta relação ou resultado.
    .orderBy(F.desc("popularidade")) #Apresenta os maiores valores primeiro.
    .limit(5) #Exibe os cinco primeiros filmes após a ordenação.
)


# 3. Quantidade de filmes por gênero
display( #Exibe o resultado do DataFrame para responder à pergunta de negócio.
    spark.table("medallion.gold.bridge_movie_genre") #Lê a tabela persistida que serve de origem para este bloco.
    .join( #Associa cada relação de gênero ao nome que será exibido na contagem.
        spark.table("medallion.gold.dim_genres"), #Lê a tabela persistida que serve de origem para este bloco.
        "sk_genre_id" #Chave substituta que identifica o gênero na dimensão e na ponte.
    )
    .groupBy("nome_genero") #Agrupa as relações por gênero para contar os filmes associados.
    .agg( #Calcula as medidas de resumo do conjunto ou dos grupos definidos.
        F.countDistinct("sk_movie_id").alias("qtd_filmes") #Conta cada filme uma única vez dentro do grupo.
    )
    .orderBy(F.desc("qtd_filmes")) #Apresenta os maiores valores primeiro.
)


# 4. Top 10 filmes por receita, com RANK()
janela_receita = Window.orderBy(F.desc("receita_usd")) #Ordena pela receita para calcular o ranking financeiro.

display( #Exibe o resultado do DataFrame para responder à pergunta de negócio.
    df_fato
    .join(df_filmes.select("sk_movie_id", "titulo"), "sk_movie_id") #Acrescenta o título aos filmes classificados por receita.
    .filter(F.col("receita_usd").isNotNull()) #Retira filmes sem receita conhecida antes de calcular o ranking.
    .withColumn("posicao", F.rank().over(janela_receita)) #Atribui a mesma posição aos valores empatados e deixa lacunas nas posições seguintes.
    .orderBy("posicao", "sk_movie_id") #Ordena os empates pela chave apenas na apresentação, sem alterar o RANK.
    .select( #Seleciona os atributos necessários para esta relação ou resultado.
        "posicao", #Posição calculada no ranking; valores empatados recebem a mesma posição.
        "titulo", #Título do filme utilizado na identificação dos resultados de negócio.
        "receita_usd", #Receita do filme em dólares, preservada da Silver.
        "receita_brl" #Receita convertida para reais na Silver com a cotação utilizada naquela carga.
    )
    .limit(10) #Exibe no máximo dez linhas, mesmo se houver empate na posição de corte.
)


# 5. Ator com mais participações nos últimos 2 anos
atores_2_anos = ( #Conta os filmes associados a cada ator no intervalo de 24 meses.
    spark.table("medallion.gold.bridge_movie_person") #Lê a tabela persistida que serve de origem para este bloco.
    .join( #Seleciona as pessoas classificadas como atores para contar suas participações.
        spark.table("medallion.gold.dim_people") #Lê a tabela persistida que serve de origem para este bloco.
        .filter(F.col("tipo_pessoa") == "Ator"),
        "sk_person_id" #Chave substituta que identifica a combinação de pessoa e tipo de atuação.
    )
    .join( #Acrescenta data e status para limitar a análise de atores ao período solicitado.
        df_filmes.select("sk_movie_id", "data_lancamento", "status_filme"), #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id" #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
    )
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.lit(data_limite)) & #Limita a análise à data de referência, incluindo o próprio dia do lançamento.
        (F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -24)) #Inclui filmes a partir de 24 meses antes da data de referência.
    )
    .groupBy("nome_pessoa") #Agrupa as participações por nome do ator.
    .agg( #Calcula as medidas de resumo do conjunto ou dos grupos definidos.
        F.countDistinct("sk_movie_id").alias("qtd_participacoes") #Conta cada filme uma única vez dentro do grupo.
    )
)

janela_atores = Window.orderBy(F.desc("qtd_participacoes")) #Ordena pela medida de negócio para identificar o primeiro lugar e seus empates.

display( #Exibe o resultado do DataFrame para responder à pergunta de negócio.
    atores_2_anos
    .withColumn("posicao", F.rank().over(janela_atores)) #Atribui a mesma posição aos valores empatados e deixa lacunas nas posições seguintes.
    .filter(F.col("posicao") == 1) #Preserva todos os empatados na primeira posição.
    .select("nome_pessoa", "qtd_participacoes") #Seleciona os atributos necessários para esta relação ou resultado.
)


# 6. Produtora com maior lucro nos últimos 5 anos
produtoras_5_anos = ( #Soma o lucro dos filmes associados a cada produtora no intervalo de 60 meses.
    spark.table("medallion.gold.bridge_movie_company") #Lê a tabela persistida que serve de origem para este bloco.
    .join( #Associa os filmes aos nomes das produtoras.
        spark.table("medallion.gold.dim_companies"), #Lê a tabela persistida que serve de origem para este bloco.
        "sk_company_id" #Chave substituta da produtora, utilizada nas relações com os filmes.
    )
    .join( #Acrescenta data e status para limitar a análise de produtoras ao período solicitado.
        df_filmes.select("sk_movie_id", "data_lancamento", "status_filme"), #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id" #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
    )
    .join( #Acrescenta o lucro de cada filme para calcular o total associado à produtora.
        df_fato.select("sk_movie_id", "lucro_usd"), #Seleciona os atributos necessários para esta relação ou resultado.
        "sk_movie_id" #Chave substituta do filme, usada para ligar dimensões, fato e tabelas ponte.
    )
    .filter(
        (F.col("status_filme") == "Lançado") &
        (F.col("data_lancamento") <= F.lit(data_limite)) & #Limita a análise à data de referência, incluindo o próprio dia do lançamento.
        (F.col("data_lancamento") >= F.add_months(F.lit(data_limite), -60)) & #Inclui filmes a partir de 60 meses antes da data de referência.
        F.col("lucro_usd").isNotNull() #Considera apenas filmes com lucro disponível para a comparação financeira.
    )
    .groupBy("nome_produtora") #Agrupa os filmes associados a cada produtora.
    .agg( #Calcula as medidas de resumo do conjunto ou dos grupos definidos.
        F.sum("lucro_usd").alias("lucro_total_usd") #Soma o lucro dos filmes associados à produtora, sem ratear coproduções.
    )
)

janela_produtoras = Window.orderBy(F.desc("lucro_total_usd")) #Ordena pela medida de negócio para identificar o primeiro lugar e seus empates.

display( #Exibe o resultado do DataFrame para responder à pergunta de negócio.
    produtoras_5_anos
    .withColumn("posicao", F.rank().over(janela_produtoras)) #Atribui a mesma posição aos valores empatados e deixa lacunas nas posições seguintes.
    .filter(F.col("posicao") == 1) #Preserva todos os empatados na primeira posição.
    .select("nome_produtora", "lucro_total_usd") #Seleciona os atributos necessários para esta relação ou resultado.
)
